# Install Packages

In [ ]:
import Pkg
Pkg.add([
    "POMDPs",
    "POMDPTools",
    "Distributions",
    "Parameters",
    "Plots",
    "ParticleFilters",
    "QMDP",
    "BasicPOMCP",
    "POMCPOW",
    "D3Trees",
])


In [ ]:
using POMDPs
using POMDPTools
using Distributions
using Parameters
using Random
using Printf


# LightDark Demo

In [ ]:
# Use the headless ("null") GR workstation so plotting works without a display
# or system OpenGL libraries -- this avoids the libGL.so error seen on Colab.
ENV["GKSwstype"] = "100"
using Plots
gr()


# Problem Definition

In [ ]:
@with_kw struct SimpleLightDark <: POMDP{Int,Int,Float64}
    discount::Float64       = 0.95
    correct_r::Float64      = 100.0
    incorrect_r::Float64    = -100.0
    light_loc::Int          = 10
    radius::Int             = 60
end

# Qualified with `POMDPs.` because POMDPTools also exports names like `actions`;
# without the prefix Julia silently creates unrelated shadow methods instead of
# extending the POMDPs.jl interface, and simulation fails with a MethodError.
POMDPs.discount(p::SimpleLightDark) = p.discount
POMDPs.isterminal(p::SimpleLightDark, s::Number) = !(s in -p.radius:p.radius)

const ACTIONS = [-10, -1, 0, 1, 10]
POMDPs.actions(p::SimpleLightDark) = ACTIONS
const ACTION_INDS = Dict(a=>i for (i,a) in enumerate(ACTIONS))
POMDPs.actionindex(p::SimpleLightDark, a::Int) = ACTION_INDS[a]

POMDPs.states(p::SimpleLightDark) = -p.radius:(p.radius + 1)
POMDPs.stateindex(p::SimpleLightDark, s::Int) = s + p.radius + 1

function POMDPs.transition(p::SimpleLightDark, s::Int, a::Int)
    if a == 0
        return SparseCat([p.radius + 1], [1.0])
    else
        return SparseCat([clamp(s + a, -p.radius, p.radius)], [1.0])
    end
end

POMDPs.observation(p::SimpleLightDark, sp::Int) = Normal(sp, abs(sp - p.light_loc) + 0.0001)

function POMDPs.reward(p::SimpleLightDark, s::Int, a::Int)
    if a == 0
        return s == 0 ? p.correct_r : p.incorrect_r
    else
        return -1.0
    end
end

function POMDPs.initialstate(p::SimpleLightDark)
    support = div(-p.radius, 2):div(p.radius, 2)
    probs = fill(1 / length(support), length(support))
    return SparseCat(support, probs)
end

p = SimpleLightDark()


# Visualization

In [ ]:
function plothist(pomdp, hist, heading="LightDark")
    tmax = 80
    smin = -10
    smax = 20
    vsh = collect(filter(s -> !isterminal(pomdp, s), state_hist(hist)[1:end-1]))
    bh = belief_hist(hist)

    pts = Int[]
    pss = Int[]
    pws = Float64[]

    for t in 0:length(bh)-1
        b = bh[t+1]
        for s in smin:smax
            w = 10.0 * sqrt(pdf(b, s))
            if 0.0 < w < 1.0
                w = 1.0
            end
            push!(pts, t)
            push!(pss, s)
            push!(pws, w)
        end
    end

    T = range(0.0, stop=tmax, length=100)
    S = range(-1.0, stop=21.0, length=100)
    inv_grays = cgrad([RGB(1.0, 1.0, 1.0), RGB(0.0, 0.0, 0.0)])
    fig = contour(T, S, (t, s) -> abs(s - pomdp.light_loc),
            bg_inside=:black,
            fill=true,
            xlim=(0, tmax),
            ylim=(smin, smax),
            color=inv_grays,
            xlabel="Time",
            ylabel="State",
            cbar=false,
            legend=:topright,
            title=@sprintf("%s (Reward: %8.2f)", heading, discounted_reward(hist))
           )
    plot!(fig, [0, tmax], [0, 0], linewidth=1, color="green", label="Goal", line=:dash)
    scatter!(fig, pts, pss, color="lightblue", label="Belief Particles", markersize=pws, marker=stroke(0.1, 0.3))
    plot!(fig, 0:length(vsh)-1, vsh, linewidth=3, color="orangered", label="Trajectory")

    return fig
end;


# Baseline: Random Policy with a Particle Filter

In [ ]:
using ParticleFilters

function run_pf_simulation(pomdp)
    rng = MersenneTwister(7)
    pf = BootstrapFilter(pomdp, 10_000; rng=rng)
    policy = RandomPolicy(pomdp; rng=rng)
    hr = HistoryRecorder(max_steps=80, rng=rng)
    h = simulate(hr, pomdp, policy, pf, initialstate(pomdp), 1)
    return plothist(pomdp, h)
end

run_pf_simulation(p)


# QMDP

In [ ]:
using QMDP

function run_qmdp_solver(pomdp)
    solver = QMDPSolver(max_iterations=100, verbose=true)
    return solve(solver, pomdp)
end

qmdp_policy = run_qmdp_solver(p)


In [ ]:
function run_qmdp_simulation(pomdp, policy)
    rng = MersenneTwister(14)
    pf = BootstrapFilter(pomdp, 10_000; rng=rng)
    hr = HistoryRecorder(max_steps=80, rng=rng)
    h = simulate(hr, pomdp, policy, pf, initialstate(pomdp), 1)
    return plothist(pomdp, h, "QMDP")
end

run_qmdp_simulation(p, qmdp_policy)


# POMCP

In [ ]:
using BasicPOMCP

function run_pomcp_solver(pomdp)
    rng = MersenneTwister(7)
    pf = BootstrapFilter(pomdp, 10_000; rng=rng)
    sol = POMCPSolver(
        max_depth=20,
        max_time=0.01,
        c=100.0,
        tree_queries=typemax(Int),
        estimate_value=FORollout(RandomSolver()),
        rng=rng,
        tree_in_info=true,
    )
    planner = solve(sol, pomdp)
    hr = HistoryRecorder(max_steps=80, rng=rng)
    h = simulate(hr, pomdp, planner, pf, initialstate(pomdp), 1)
    return planner, plothist(pomdp, h, "POMCP")
end

pomcp_planner, pomcp_fig = run_pomcp_solver(p)
pomcp_fig


In [ ]:
using D3Trees

b0 = initialstate(p)
a, info = action_info(pomcp_planner, b0)
D3Tree(info[:tree], init_expand=2)


# POMCPOW

In [ ]:
using POMCPOW

function run_pomcpow_solver(pomdp)
    rng = MersenneTwister(7)
    pf = BootstrapFilter(pomdp, 10_000; rng=rng)
    sol = POMCPOWSolver(
        tree_queries=10_000_000,
        criterion=MaxUCB(90.0),
        max_depth=20,
        max_time=0.01,
        enable_action_pw=false,
        k_observation=5.0,
        alpha_observation=1/15.0,
        estimate_value=FORollout(RandomSolver()),
        check_repeat_obs=false,
        tree_in_info=true,
        rng=rng,
    )
    planner = solve(sol, pomdp)
    hr = HistoryRecorder(max_steps=80, rng=rng)
    h = simulate(hr, pomdp, planner, pf, initialstate(pomdp), 1)
    return planner, plothist(pomdp, h, "POMCPOW")
end

pomcpow_planner, pomcpow_fig = run_pomcpow_solver(p)
pomcpow_fig


In [ ]:
b0 = initialstate(p)
a, info = action_info(pomcpow_planner, b0)
D3Tree(info[:tree], init_expand=2)
